# 04 — NVIDIA Triton Model Serving & Inference

## Objective
Understand how NVIDIA Triton Inference Server deploys and serves machine learning models. Explore model repository structure, model versioning, and configuration using `config.pbtxt`. Deploy the exported BERT ONNX model, perform inference through a Triton client, and validate server responses against local ONNX Runtime predictions.

### Understand why an inference server is needed.

Inference server is required to serve a deployed dedicated model from the API to other resources, etc using onnx format, where is acceptable for all providers. It may also reduce a self-hosted approach using dedicated NVIDIA Triton Inference Server solution as containerized model environment.

### Explore how NVIDIA Triton organizes models and their versions.

#### Create a model repository

Expected structure of the repository for Triton should looks like this:
```plaintext
model_repository/
├── text_detection
│   ├── 1 
│   │   └── model.onnx
│   └── config.pbtxt
└── text_recognition
    ├── 1
    │   └── model.onnx
    └── config.pbtxt
```

Where `1` describe a version of the model for versioning process.

`text_detection` and `text_recognition` are examples as the names of the models.

### Define model configuration using config.pbtxt.

The next thing to understand and configure is a model `config.pbtxt` configuration file. It defines how we want to serve our model using inference server.

From the `03_onnx_inference.ipynb` notebook we can read a model session's input and output: name, shape and type:
```plaintext
input_ids ['batch_size', 'sequence_length'] tensor(int64)
attention_mask ['batch_size', 'sequence_length'] tensor(int64)
token_type_ids ['batch_size', 'sequence_length'] tensor(int64)

logits ['batch_size', 2] tensor(float)
``` 
It will help to configure the `config.pbtxt` file.

In this scenario, the configuration looks like this:
```plaintext
name: "bert_sst2"
backend: "onnxruntime"
max_batch_size : 8
input [
  {
    name: "input_ids"
    data_type: TYPE_INT64
    dims: [ -1 ]
  },
  {
    name: "attention_mask"
    data_type: TYPE_INT64
    dims: [ -1 ]
  },
  {
    name: "token_type_ids"
    data_type: TYPE_INT64
    dims: [ -1 ]
  }
]
output [
  {
    name: "logits"
    data_type: TYPE_FP32
    dims: [ 2 ]
  }
]
```

**Why `[-1]` as dims parameter?**

ONNX model expects an input shape value like [batch_size, sequence_length]. However, when the `max_batch_size > 0`, Triton handles the batch size automatically. So in the `confib.pbtxt` file we declare dims as `[-1]`, which means that, the length of sequence may change. 

For instance, for batch which consists of 4 sequences and the length of 128 tokens, the actual input tensor may have the shape like `[4, 128]`.

### Run Triton Inference Server and deploy the exported ONNX model.

First pull the Triton Docker image below:

In [ ]:
!docker pull nvcr.io/nvidia/tritonserver:26.09-py3

To run Triton Inference Server with GPU support, we need to configure Docker to access NVIDIA GPUs. On supported Linux systems, this typically requires the [NVIDIA Container toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/index.html).

In [ ]:
!docker run --gpus=1 --rm \
  -p 8000:8000 \
  -p 8001:8001 \
  -p 8002:8002 \
  -v "C:/Users/PC/source/repos/BERT-Inference-Lab/models/model_repository:/models" \
  nvcr.io/nvidia/tritonserver:26.09-py3 \
  tritonserver --model-repository=/models

The CPU option without installing a NVIDIA Container toolkit:

In [ ]:
!docker run --rm \
  -p 8000:8000 \
  -p 8001:8001 \
  -p 8002:8002 \
  -v "C:/Users/PC/source/repos/BERT-Inference-Lab/models/model_repository:/models" \
  nvcr.io/nvidia/tritonserver:26.09-py3 \
  tritonserver --model-repository=/models

To verify that Triton Inference Server is ready, we can use the `/v2/health/ready` endpoint. To check whether a specific model is ready for inference, we can use `/v2/models/bert_sst2/ready`.

In [ ]:
curl -v localhost:8000/v2/health/ready

In [ ]:
curl -v localhost:8000/v2/models/bert_sst2/ready

### Send inference requests and retrieve predictions using a Triton client.

When we launched and tested a Triton server with BERT model, we can start sending messages to it. In this scenario from three allowed ways of using:
- HTTP(S) API
- gRPC API
- NATVICE C API

I will use a HTTP(S) API approach.

Install HTTP triton client

In [ ]:
%pip install tritonclient[http]

In [2]:
import numpy as np
import tritonclient.http as httpclient
from transformers import AutoTokenizer

MODEL_NAME = "bert_sst2"
TOKENIZER_NAME = "textattack/bert-base-uncased-SST-2"

client = httpclient.InferenceServerClient(url="localhost:8000")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def bert_function(text: str):
    encoded = tokenizer(
        text,
        return_tensors="np",
        padding="max_length",
        truncation=True,
        max_length=128
    )

    input_ids = encoded["input_ids"].astype(np.int64)
    attention_mask = encoded["attention_mask"].astype(np.int64)
    token_type_ids = encoded["token_type_ids"].astype(np.int64)

    inputs = []

    for name, data in [
        ("input_ids", input_ids),
        ("attention_mask", attention_mask),
        ("token_type_ids", token_type_ids)
    ]:
        triton_input = httpclient.InferInput(
            name,
            data.shape,
            datatype="INT64"
        )
        triton_input.set_data_from_numpy(data)
        inputs.append(triton_input)
    
    outputs = [
        httpclient.InferRequestedOutput("logits")
    ]

    result = client.infer(
        model_name=MODEL_NAME,
        inputs=inputs,
        outputs=outputs
    )

    logits = result.as_numpy("logits")

    # SST-2: class 0 = negative, class 1 = positive
    prediction = int(np.argmax(logits, axis=-1)[0])
    label = "positive" if prediction == 1 else "negative"

    return {
        "text": text,
        "logits": logits,
        "prediction": prediction,
        "label": label
    }

Test the solution

In [ ]:
result = bert_function("This movie was absolutely fantasitc!")

print("Text:", result["text"])
print("Logits:", result["logits"])
print("Prediction:", result["prediction"])
print("Sentiment:", result["label"])

### Validate server responses against local inference results.

In [5]:
from pathlib import Path

import onnxruntime as ort


MODEL_DIR = Path.cwd().parent / "models" / "bert-sst2-onnx"
ONNX_PATH = MODEL_DIR / "model.onnx"

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

input_names = [item.name for item in session.get_inputs()]

def bert_onnx_function(text: str):
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="np"
    )

    ort_inputs = {
        name: encoded[name].astype(np.int64)
        for name in input_names
    }

    outputs = session.run(
        None,
        ort_inputs
    )
    
    logits = np.asarray(outputs[0], dtype=np.float32)

    prediction = int(np.argmax(logits, axis=-1)[0])
    label = "positive" if prediction == 1 else "negative"

    return {
        "text": text,
        "logits": logits,
        "prediction": prediction,
        "label": label      
    }


#### Compare triton result with an ONNX runtime result

In [6]:
text = "This movie was absolutely fantastic!"

onnx_result = bert_onnx_function(text)
triton_result = bert_function(text)

print("ONNX Runtime:", onnx_result["logits"])
print("Triton:", triton_result["logits"])

print(
    "Max absolute difference:",
    np.max(np.abs(onnx_result["logits"] - triton_result["logits"]))
)

print("ONNX prediction:", onnx_result["label"])
print("Triton prediction:", triton_result["label"])

ONNX Runtime: [[-4.1500936  3.9333801]]
Triton: [[-4.1500936  3.9333801]]
Max absolute difference: 0.0
ONNX prediction: positive
Triton prediction: positive


### Multi-Text Inference Consistency Test

In [7]:
import numpy as np

test_texts = [
    "This movie was absolutely fantastic!",
    "This movie was boring and disappointing.",
    "I really enjoyed this film.",
    "The acting was terrible.",
    "An excellent and entertaining story.",
    "I would not recommend this movie.",
    "The film was okay, nothing special.",
    "One of the worst movies I have ever seen.",
    "The performances were brilliant.",
    "I loved every minute of it."
]

results = []

for text in test_texts:
    onnx_result = bert_onnx_function(text)
    triton_result = bert_function(text)

    max_diff = float(np.max(np.abs(
        onnx_result["logits"] - triton_result["logits"]
    )))

    predictions_match = (
        onnx_result["prediction"] == triton_result["prediction"]
    )

    results.append({
        "text": text,
        "onnx_label": onnx_result["label"],
        "triton_label": triton_result["label"],
        "predictions_match": predictions_match,
        "max_abs_diff": max_diff
    })

for row in results:
    print(f'Text: {row["text"]}')
    print(f'ONNX: {row["onnx_label"]}')
    print(f'Triton: {row["triton_label"]}')
    print(f'Predictions match: {row["predictions_match"]}')
    print(f'Max abs difference: {row["max_abs_diff"]:.8f}')
    print()

Text: This movie was absolutely fantastic!
ONNX: positive
Triton: positive
Predictions match: True
Max abs difference: 0.00000000

Text: This movie was boring and disappointing.
ONNX: negative
Triton: negative
Predictions match: True
Max abs difference: 0.00000000

Text: I really enjoyed this film.
ONNX: positive
Triton: positive
Predictions match: True
Max abs difference: 0.00000000

Text: The acting was terrible.
ONNX: negative
Triton: negative
Predictions match: True
Max abs difference: 0.00000000

Text: An excellent and entertaining story.
ONNX: positive
Triton: positive
Predictions match: True
Max abs difference: 0.00000000

Text: I would not recommend this movie.
ONNX: negative
Triton: negative
Predictions match: True
Max abs difference: 0.00000000

Text: The film was okay, nothing special.
ONNX: negative
Triton: negative
Predictions match: True
Max abs difference: 0.00000000

Text: One of the worst movies I have ever seen.
ONNX: negative
Triton: negative
Predictions match: True


In [8]:
all_predictions_match = all(
    row["predictions_match"] for row in results
)

max_diff = max(
    row["max_abs_diff"] for row in results
)

print(f"Number of texts: {len(results)}")
print(f"All predictions match: {all_predictions_match}")
print(f"Maximum absolute difference: {max_diff:.8f}")

Number of texts: 10
All predictions match: True
Maximum absolute difference: 0.00000000


### Results

- Tested 10 text inputs.
- All ONNX Runtime and Triton predictions matched.
- Maximum absolute difference between logits: 0.0.
- Triton successfully served the ONNX model through its HTTP inference API.

The test confirmed inference consistency between local ONNX Runtime execution and Triton serving for the selected inputs.